# Tutoriel — Suivi des retards Rouen Rive Droite → Paris Saint-Lazare

Ce notebook déroule **toutes les étapes du pipeline**, une par une, pour comprendre le code :

1. **Configuration** — chargement des paramètres (`Settings`).
2. **Client API SNCF** — récupération des départs temps réel à Rouen Rive Droite.
3. **Stockage** — snapshots JSONL journaliers et déduplication.
4. **Flux GTFS-RT VehiclePositions** — positions GPS temps réel des trains (odométrie).
5. **Détection d'arrêts** — estimer l'arrivée réelle d'un train à partir de sa trace GPS.
6. **Rapports** — agrégats journaliers / hebdomadaires / mensuels.
7. **Site** — génération des fichiers JSON consommés par la page GitHub Pages.

> Le notebook fonctionne **sans token API** : il utilise des fixtures enregistrées (réponses SNCF simulées) et, pour les parties réseau, des exemples optionnels que vous pouvez activer.
> Lancer avec : `jupyter notebook notebooks/tutorial.ipynb` depuis la racine du dépôt (le chemin `backend/` est ajouté automatiquement).

In [1]:
import sys, os
from pathlib import Path

# Rendre le package importable depuis la racine du dépôt
ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))
sys.path.insert(0, str(ROOT / "tests"))

from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

TZ = ZoneInfo("Europe/Paris")
print("racine du dépôt :", ROOT)

racine du dépôt : /workspace/github__joubertdamien__rouenparis


## 1. Configuration

Toute la configuration passe par `rouenparis.config.Settings` : valeurs par défaut + variables d'environnement. Les gares par défaut sont Rouen Rive Droite (`stop_area:SNCF:87411017`) et Paris Saint-Lazare (`stop_area:SNCF:87271007`).

In [2]:
from rouenparis.config import Settings

settings = Settings(api_token=os.environ.get("SNCF_API_TOKEN", "<token>"))
print("origine      :", settings.origin_stop_area)
print("destination  :", settings.destination_stop_area)
print("fenêtre      :", settings.lookahead_minutes, "min")
print("seuil retard :", settings.min_delay_minutes, "min")
print("flux GTFS-RT :", settings.gtfs_rt_vehicle_positions_url)
print("suivi GPS    :", settings.track_positions)
print("gare (odométrie):", settings.station_latitude, settings.station_longitude)

origine      : stop_area:SNCF:87411017
destination  : stop_area:SNCF:87271007
fenêtre      : 240 min
seuil retard : 5 min
flux GTFS-RT : https://proxy.transport.data.gouv.fr/resource/sncf-gtfs-rt-vehicle-positions
suivi GPS    : True
gare (odométrie): 48.8809 2.3206


## 2. Client API SNCF — départs temps réel

`SncfClient.fetch_departures()` interroge `https://api.sncf.com/v1/coverage/sncf/stop_areas/<origine>/departures` avec `data_freshness=realtime`, puis pour chaque départ remonte le `vehicle_journey` pour trouver l'**arrivée à Paris Saint-Lazare** et calculer le retard à l'arrivée.

La cellule suivante utilise une **fixture enregistrée** (pas d'appel réseau). Pour un vrai appel, remplacez `FakeOpener` par rien et appelez `SncfClient(settings).fetch_departures(datetime.now(tz=TZ))` avec votre token.

In [3]:
from fixtures import FakeOpener, departures_response_for
from rouenparis.sncf_client import SncfClient

demo_day = date(2026, 10, 15)
client = SncfClient(settings, opener=FakeOpener(departures_response_for(demo_day)))
trains = client.fetch_departures(datetime(2026, 10, 15, 6, 0, tzinfo=TZ))

for train in trains:
    dep_delay = (train.departure_delay_seconds or 0) / 60
    arr_delay = (train.arrival_delay_seconds / 60) if train.arrival_delay_seconds is not None else None
    arrival = f"{train.base_arrival:%H:%M}" if train.base_arrival else "—"
    print(f"train {train.train_number:>6} ({train.line_code})  départ {train.base_departure:%H:%M} +{dep_delay:.0f} min   "
          f"arrivée prévue {arrival}   retard arrivée {arr_delay} min   statut={train.status or '—'}")

train   3123 (K22)  départ 07:12 +0 min   arrivée prévue 08:30   retard arrivée 0.0 min   statut=—
train   3125 (K22)  départ 07:30 +12 min   arrivée prévue 08:48   retard arrivée 19.0 min   statut=—
train   3401 (K24)  départ 08:05 +0 min   arrivée prévue —   retard arrivée None min   statut=cancelled


### Anatomie d'une observation

Chaque départ devient un `TrainObservation` sérialisable — c'est exactement ce qui est écrit dans les snapshots.

In [4]:
import json
print(json.dumps(trains[1].to_dict(), indent=2, ensure_ascii=False))

{
  "train_number": "3125",
  "line_code": "K22",
  "line_name": "TER 3125",
  "direction": "Paris Saint-Lazare",
  "base_departure": "2026-10-15T07:30:00+02:00",
  "realtime_departure": "2026-10-15T07:42:00+02:00",
  "departure_delay_seconds": 720,
  "departure_state": "realtime",
  "base_arrival": "2026-10-15T08:48:00+02:00",
  "realtime_arrival": "2026-10-15T09:07:00+02:00",
  "arrival_delay_seconds": 1140,
  "arrival_state": "realtime",
  "status": "",
  "data_freshness": "realtime",
  "collected_at": "2026-10-28T11:55:26.054520+01:00"
}


## 3. Stockage — snapshots et déduplication

Chaque collecte écrit une ligne JSONL dans `data/snapshots/YYYY-MM-DD.jsonl`. Comme on collecte toutes les 15 minutes, un même train est observé plusieurs fois : `dedupe_final_observations()` garde pour chaque train **la meilleure observation** (arrivée temps réel connue > retard connu > observation la plus tardive).

In [5]:
import tempfile
from rouenparis.storage import Snapshot, append_snapshot, load_snapshots
from rouenparis.storage import dedupe_final_observations

data_dir = Path(tempfile.mkdtemp())

# Deux collectes le même jour : le matin et le soir
for hour in (7, 21):
    snapshot = Snapshot(collected_at=datetime(2026, 10, 15, hour, 30, tzinfo=TZ), trains=trains)
    append_snapshot(data_dir, snapshot)

snapshots = list(load_snapshots(data_dir))
print(f"{len(snapshots)} snapshots, {sum(len(s.trains) for s in snapshots)} observations brutes")

final = dedupe_final_observations(snapshots)
print(f"{len(final)} trains après déduplication")
for train in final:
    print(f"  train {train.train_number}: {train.departure_state} -> retard arrivée {train.arrival_delay_seconds} s")

2 snapshots, 6 observations brutes
3 trains après déduplication
  train 3123: realtime -> retard arrivée 0 s
  train 3125: realtime -> retard arrivée 1140 s
  train 3401: realtime -> retard arrivée None s


## 4. Flux GTFS-RT VehiclePositions — positions GPS des trains

La SNCF publie un flux **GTFS-RT VehiclePositions** (protobuf) sur transport.data.gouv.fr, rafraîchi environ toutes les 30–60 s. Deux bonnes nouvelles :

- **aucun token requis** ;
- le décodage protobuf est fait **sans dépendance** par `rouenparis.gtfsrt_proto` (lecture manuelle du wire-format).

La cellule suivante construit un flux GTFS-RT synthétique à la main (même construction binaire qu'un vrai flux) et le décode.

In [6]:
from test_gtfsrt_proto import build_feed, build_vehicle_entity
from rouenparis.gtfsrt_proto import parse_feed

payload = build_feed([
    build_vehicle_entity("entity-1", "OCESN3125F", "OCE:Train:1", 48.8809, 2.3206,
                         speed=0.0, status=1, stop_id="StopArea:OCE87271007",
                         timestamp=1760767680, vehicle_id="TGV-3125"),
    build_vehicle_entity("entity-2", "OCESN850627F", "OCE:Train:2", 49.2, 1.5,
                         speed=45.0, status=0, timestamp=1760767680, vehicle_id="TGV-850627"),
])

feed = parse_feed(payload)
print(f"version={feed.version}  timestamp={feed.timestamp}  véhicules={len(feed.vehicles)}")
for vehicle in feed.vehicles:
    print(f"  {vehicle.vehicle_id}  trip={vehicle.trip_id}  "
          f"({vehicle.position.latitude:.4f}, {vehicle.position.longitude:.4f})  "
          f"vitesse={vehicle.position.speed} m/s  statut={vehicle.current_status}  arrêt={vehicle.stop_id}")

version=2.0  timestamp=1760000000  véhicules=2
  TGV-3125  trip=OCESN3125F  (48.8809, 2.3206)  vitesse=0.0 m/s  statut=STOPPED_AT  arrêt=StopArea:OCE87271007
  TGV-850627  trip=OCESN850627F  (49.2000, 1.5000)  vitesse=45.0 m/s  statut=IN_TRANSIT_TO  arrêt=


### Vrai appel réseau (optionnel)

Décommentez pour interroger le vrai flux (aucun token nécessaire) :

```python
from rouenparis.vehicle_positions import VehiclePositionsClient
records = VehiclePositionsClient(settings).fetch_vehicles_for_route()
print(len(records), "positions de trains")
```

## 5. Odyssée d'un train — détection d'arrêts et arrivée estimée

`rouenparis.odometry` transforme une série de positions GPS en :

- **distance parcourue** (haversine entre fixes consécutifs) ;
- **temps de trajet** ;
- **événements d'arrêt** : un arrêt = un run maximal de fixes immobiles (dans un rayon de 150 m) pendant au moins 60 s. Les trous dans le flux (> 3 min) séparent deux arrêts.

Le **milieu de l'arrêt** à Paris Saint-Lazare est notre meilleure estimation de l'arrivée réelle — indépendante de l'annonce API, ce qui permet de **croiser les deux sources**.

In [7]:
from rouenparis.odometry import Fix, compute_odometry, estimate_arrival_at, haversine_m

ROUEN = (49.4431, 1.0993)
PSTL = (48.8809, 2.3206)  # Paris Saint-Lazare
print(f"distance Rouen → Paris à vol d'oiseau : {haversine_m(*ROUEN, *PSTL) / 1000:.1f} km")

# Simulation : départ de Rouen à 07:30, trajet de 78 min, arrêt à Paris 08:48 → 09:08
fixes = []
start = datetime(2026, 10, 15, 7, 30, tzinfo=TZ)
for i in range(78):
    frac = i / 78
    fixes.append(Fix(start + timedelta(minutes=i),
                     ROUEN[0] + (PSTL[0] - ROUEN[0]) * frac,
                     ROUEN[1] + (PSTL[1] - ROUEN[1]) * frac, speed=30.0))
for i in range(21):
    fixes.append(Fix(start + timedelta(minutes=78 + i), PSTL[0] + 0.0001, PSTL[1],
                     speed=0.0, stop_id="StopArea:OCE87271007"))

odometry = compute_odometry("OCESN3125F", fixes)
print(f"distance totale : {odometry.distance_m / 1000:.1f} km")
print(f"temps de trajet : {odometry.moving_time_seconds / 60:.0f} min")
print(f"arrêts détectés  : {len(odometry.stop_events)}")
for stop in odometry.stop_events:
    print(f"  arrêt {stop.start:%H:%M} → {stop.end:%H:%M}  (durée {stop.duration_seconds:.0f} s, arrêt flux={stop.feed_stop_id})")

distance Rouen → Paris à vol d'oiseau : 108.6 km
distance totale : 108.6 km
temps de trajet : 78 min
arrêts détectés  : 1
  arrêt 08:48 → 09:08  (durée 1200 s, arrêt flux=StopArea:OCE87271007)


In [8]:
estimated_arrival = estimate_arrival_at(odometry, PSTL[0], PSTL[1])
scheduled_arrival = datetime(2026, 10, 15, 8, 48, tzinfo=TZ)
print(f"arrivée estimée par odométrie : {estimated_arrival:%H:%M:%S}")
print(f"arrivée prévue                : {scheduled_arrival:%H:%M:%S}")
print(f"retard odométrie              : {(estimated_arrival - scheduled_arrival).total_seconds() / 60:+.1f} min")

# L'API annonçait +19 min (arrivée 09:07) : l'odométrie donne +10,5 min.
# C'est exactement ce croisement que le site affiche dans les colonnes « Arrêt odométrie » et « Retard odométrie ».
print(f"(API : +19,0 min — écart API/odométrie : {19.0 - (estimated_arrival - scheduled_arrival).total_seconds() / 60:+.1f} min)")

arrivée estimée par odométrie : 08:58:00
arrivée prévue                : 08:48:00
retard odométrie              : +10.0 min
(API : +19,0 min — écart API/odométrie : +9.0 min)


### Rapprochement course ↔ train

Les `trip_id` GTFS-RT SNCF contiennent le **numéro commercial** du train (ex. `OCESN3125F` → `3125`), identique au `headsign` Navitia. C'est la clé de jointure principale ; un repli par proximité horaire (±45 min du départ prévu) couvre les cas limites.

In [9]:
from rouenparis.positions import train_number_from_trip_id, match_odometry_to_train

print(train_number_from_trip_id("OCESN3125F"), train_number_from_trip_id("OCE850627R"))

matched = match_odometry_to_train({"OCESN3125F": odometry}, "3125", base_departure=datetime(2026, 10, 15, 7, 30, tzinfo=TZ))
print("train 3125 rapproché :", matched is odometry)

3125 850627
train 3125 rapproché : True


## 6. Rapports — journalier, hebdomadaire, mensuel

`rouenparis.reports` transforme les snapshots dédupliqués en `TrainRecord`, agrège par période (jour / semaine ISO / mois) et calcule : ponctualité (seuil 5 min), retard moyen/max, suppressions — **avec ou sans** la vérification odométrique.

In [10]:
from rouenparis.reports import build_train_records, build_report, daily_series

records = build_train_records(snapshots, settings.min_delay_minutes * 60)
for record in records:
    print(f"{record.service_date}  train {record.train_number}  prévu {record.scheduled_departure[11:16]}  "
          f"retard {record.delay_seconds} s  statut={record.status}")

daily = build_report(records, "daily", date(2026, 10, 15), settings)
print("\nRésumé du 2026-10-15 :", json.dumps(daily["summary"], ensure_ascii=False))

2026-10-15  train 3123  prévu 07:12  retard 0 s  statut=scheduled
2026-10-15  train 3125  prévu 07:30  retard 1140 s  statut=scheduled
2026-10-15  train 3401  prévu 08:05  retard 0 s  statut=cancelled

Résumé du 2026-10-15 : {"scheduled": 3, "circulated": 2, "cancelled": 1, "measured": 2, "delayed": 1, "punctuality_rate": 50.0, "avg_delay_minutes": 9.5, "max_delay_minutes": 19.0, "min_delay_threshold_minutes": 5}


In [11]:
weekly = build_report(records, "weekly", date(2026, 10, 12), settings)
monthly = build_report(records, "monthly", date(2026, 10, 1), settings)
print("semaine  :", weekly["period"], "->", weekly["summary"]["punctuality_rate"], "% de ponctualité")
print("mois     :", monthly["period"], "->", monthly["summary"]["punctuality_rate"], "% de ponctualité")

for entry in daily_series(records, settings.min_delay_minutes * 60):
    print(f"série {entry['date']}: ponctualité={entry['punctuality_rate']}%  retard moyen={entry['avg_delay_minutes']} min")

semaine  : {'type': 'weekly', 'start': '2026-10-12', 'end': '2026-10-18'} -> 50.0 % de ponctualité
mois     : {'type': 'monthly', 'start': '2026-10-01', 'end': '2026-10-31'} -> 50.0 % de ponctualité
série 2026-10-15: ponctualité=50.0%  retard moyen=9.5 min


## 7. Génération du site (GitHub Pages)

`generate_all()` écrit dans `site-build/` :

- `index.json` — liste des rapports J/W/M + bloc odométrie ;
- `series.json` — série temporelle de ponctualité journalière ;
- `odometry.json` — par course : fixes GPS, distance, arrêts détectés ;
- `reports/daily-*.json`, `reports/weekly-*.json`, `reports/monthly-*.json` — rapports complets avec, pour chaque train, le retard annoncé **et** le retard estimé par odométrie.

Le workflow `pages.yml` copie ensuite `site/` (HTML/CSS/JS statiques) dans ce répertoire et publie le tout sur GitHub Pages.

In [12]:
from rouenparis.positions import append_position_records, load_position_records, build_odometries
from rouenparis.reports import generate_all

site_dir = Path(tempfile.mkdtemp()) / "site-build"

# On injecte la trace GPS simulée dans le stockage, comme le fait `collect-positions`
records = [
    {
        "collected_at": fix.timestamp.isoformat(),
        "entity_id": f"entity-OCESN3125F-{int(fix.timestamp.timestamp())}",
        "trip_id": "OCESN3125F",
        "route_id": "OCE:Train:1",
        "direction_id": 0,
        "stop_id": fix.stop_id,
        "current_stop_sequence": 1,
        "current_status": "STOPPED_AT" if fix.speed == 0 else "IN_TRANSIT_TO",
        "timestamp": int(fix.timestamp.timestamp()),
        "position": {"latitude": fix.latitude, "longitude": fix.longitude, "bearing": 90.0, "speed": fix.speed},
        "vehicle_id": "TGV-3125",
        "vehicle_label": "3125",
    }
    for fix in fixes
]
append_position_records(data_dir, records)

demo_settings = Settings(api_token="demo", data_dir=str(data_dir), site_dir=str(site_dir))
index = generate_all(data_dir, site_dir, demo_settings)

print("fichiers générés :")
for path in sorted(site_dir.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(site_dir), f"({path.stat().st_size} octets)")

fichiers générés :
   index.json (1888 octets)
   latest.json (87 octets)
   odometry.json (25351 octets)
   reports/daily-2026-10-15.json (2397 octets)
   reports/monthly-2026-10-01.json (2399 octets)
   reports/weekly-2026-10-12.json (2398 octets)
   series.json (394 octets)


In [13]:
report = json.loads((site_dir / "reports" / "daily-2026-10-15.json").read_text(encoding="utf-8"))
print(json.dumps(report["summary"], indent=2, ensure_ascii=False))

print("\nTrain 3125 — annonce API vs odométrie :")
train = next(t for t in report["trains"] if t["train_number"] == "3125")
print(json.dumps({k: train[k] for k in ("scheduled_arrival", "actual_arrival", "delay_minutes",
                                        "odometry_arrival", "odometry_delay_minutes",
                                        "odometry_distance_m", "odometry_stop_events")},
                 indent=2, ensure_ascii=False))

odometry_index = json.loads((site_dir / "odometry.json").read_text(encoding="utf-8"))
print("\nbloc odométrie de index.json :", json.dumps(index["odometry"], ensure_ascii=False))
print("courses suivies :", list(odometry_index["trips"].keys()))

{
  "scheduled": 3,
  "circulated": 2,
  "cancelled": 1,
  "measured": 2,
  "delayed": 1,
  "punctuality_rate": 50.0,
  "avg_delay_minutes": 9.5,
  "max_delay_minutes": 19.0,
  "min_delay_threshold_minutes": 5
}

Train 3125 — annonce API vs odométrie :
{
  "scheduled_arrival": "2026-10-15T08:48:00+02:00",
  "actual_arrival": "2026-10-15T09:07:00+02:00",
  "delay_minutes": 19.0,
  "odometry_arrival": "2026-10-15T08:58:00+02:00",
  "odometry_delay_minutes": 10.0,
  "odometry_distance_m": 108594.4,
  "odometry_stop_events": 1
}

bloc odométrie de index.json : {"trips_tracked": 1, "trains_matched": 3, "arrivals_cross_checked": 3}
courses suivies : ['OCESN3125F']


## 8. Récapitulatif — commandes équivalentes

| Étape du notebook | Commande CLI |
|---|---|
| 2. Départs temps réel | `python -m rouenparis collect` |
| 3. Snapshots | (écrit dans `data/snapshots/`) |
| 4. Positions GPS | `python -m rouenparis collect-positions` (aucun token) |
| 6–7. Rapports + site | `python -m rouenparis report` |
| Tout d'un coup | `python -m rouenparis all` |
| Tests | `PYTHONPATH=backend:tests python -m unittest discover -s tests -v` |

En production, tout est orchestré par GitHub Actions : `collect.yml` (cron 15 min) → `pages.yml` (déploiement GitHub Pages).